In [3]:
import string
import os
import random
import re
import webbrowser
import urllib.parse
import speech_recognition as sr
import wikipedia
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('stopwords')
wikipedia.set_lang("ru")

def speak(text):
    print(f"JARVIS: {text}")
    safe_text = text.replace("'", "")
    cmd = f'powershell -Command "Add-Type -AssemblyName System.Speech; $speak = New-Object System.Speech.Synthesis.SpeechSynthesizer; $speak.Speak(\'{safe_text}\')"'
    os.system(cmd)

useful_facts = [
    "Мозг человека потребляет около 20 процентов всей энергии организма, хотя составляет всего 2 процента от его массы.",
    "Векторная модель Word2Vec превращает слова в числа так, чтобы похожие по смыслу понятия находились близко в многомерном пространстве.",
    "Самый старый из ныне известных университетов — Болонский университет, он был основан в 1088 году.",
    "Первым программистом в истории считается Ада Лавлейс, написавшая алгоритм для аналитической машины Чарльза Бэббиджа в 1843 году.",
    "Скорость света в вакууме составляет примерно 300 тысяч километров в секунду.",
    "Графен — самый прочный материал, известный науке."
]

knowledge_base = {
    "привет здравствуй хай добрый день": "Здравствуйте! О великий господин, рада служить вам.",
    "как твои дела как поживаешь": "Хвала богам и вашим молитвам, у меня все отлично! Готова к работе.",
    "кто ты такая что ты умеешь": "Я ваша покорная слуга, светлейший господин! Я умею искать информацию на Википедии, открывать YouTube и делиться фактами.",
    "что такое искусственный интеллект нейросеть": "Искусственный интеллект — это область компьютерных наук, создающая системы, способные решать задачи, требующие человеческого разума.",
    "как работает интернет сетевые протоколы": "Интернет работает за счет передачи пакетов данных по протоколам TCP IP между маршрутизаторами и серверами.",
    "что такое база данных sql": "База данных — это организованная система хранения информации, а SQL — язык для выполнения запросов к ней.",
    "какая погода на улице": "Благодаря вашей улыбке, о прекрасный госрлдин, весь мир померк!",
    "спасибо благодарю": "Не благодарите меня о великий ум, я покорный раб помогу с чем смогу!",
    "пока до свидания отключить": "Пока, прощай мой прекрасный светлый господин!"
}

my_stopwords = set(stopwords.words('russian'))

def preprocess_text(text):
    tokens = word_tokenize(text.lower(), language='russian')
    cleaned = [word for word in tokens if word not in string.punctuation and word not in my_stopwords]
    return " ".join(cleaned)

def get_best_response(user_input):
    cleaned_input = preprocess_text(user_input)
    if not cleaned_input:
        return "Извините, я не совсем вас поняла."

    questions = list(knowledge_base.keys())
    cleaned_questions = [preprocess_text(q) for q in questions]

    vectorizer = TfidfVectorizer().fit(cleaned_questions + [cleaned_input])
    q_vectors = vectorizer.transform(cleaned_questions)
    input_vector = vectorizer.transform([cleaned_input])

    similarities = cosine_similarity(input_vector, q_vectors)[0]
    best_match_index = similarities.argmax()

    if similarities[best_match_index] < 0.2:
        return "Я пока не знаю ответа на этот вопрос, но вы можете попросить меня найти его на Википедии!"

    matched_question = questions[best_match_index]
    return knowledge_base[matched_question]

def clean_wikipedia_query(text):
    stop_phrases = [
        r'\bнайди\b', r'\bпоищи\b', r'\bищи\b', r'\bоткрой\b', 
        r'\bна википедии\b', r'\bв википедии\b', r'\bвикипедия\b', r'\bвикипедию\b', r'\bвикипедии\b',
        r'\bпро\b', r'\bо\b', r'\bоб\b', r'\bинформацию\b', r'\bстатью\b'
    ]
    query = text.lower()
    for pattern in stop_phrases:
        query = re.sub(pattern, '', query)
    return query.strip()

def open_wikipedia_page(query):
    try:
        search_results = wikipedia.search(query)
        if search_results:
            best_match = search_results[0]
            page = wikipedia.page(best_match, auto_suggest=False)
            speak(f"Нашла статью: {page.title}")
            webbrowser.open(page.url)
        else:
            speak("К сожалению, я ничего не нашла по этому запросу.")
    except Exception:
        encoded_query = urllib.parse.quote(query)
        speak(f"Ищу на Википедии: {query}")
        webbrowser.open(f"https://ru.wikipedia.org/w/index.php?search={encoded_query}")

recognizer = sr.Recognizer()

def listen():
    with sr.Microphone() as source:
        print("\nСлушаю...")
        recognizer.dynamic_energy_threshold = False
        recognizer.energy_threshold = 300

        recognizer.pause_threshold = 1.8        
        recognizer.non_speaking_duration = 0.8
        
        recognizer.adjust_for_ambient_noise(source, duration=0.8)
        try:
            audio = recognizer.listen(source, timeout=6, phrase_time_limit=10)
            text = recognizer.recognize_google(audio, language='ru-RU')
            print(f"Вы: {text}")
            return text
        except (sr.WaitTimeoutError, sr.UnknownValueError):
            return None
        except sr.RequestError:
            speak("Ошибка сервиса распознавания речи.")
            return None

speak("Система активирована. Чем могу помочь?")

while True:
    user_command = listen()
    
    if user_command:
        cmd_lower = user_command.lower()
        
        if "википед" in cmd_lower:
            search_query = clean_wikipedia_query(cmd_lower)
            if search_query:
                open_wikipedia_page(search_query)
            else:
                speak("Открываю главную страницу Википедии.")
                webbrowser.open("https://ru.wikipedia.org")
        elif "ютуб" in cmd_lower or "youtube" in cmd_lower:
            speak("Открываю YouTube.")
            webbrowser.open("https://www.youtube.com")
        elif "факт" in cmd_lower or "расскажи что-то" in cmd_lower:
            speak(random.choice(useful_facts))
        else:
            response = get_best_response(user_command)
            speak(response)
        
        if any(word in cmd_lower for word in ["отключить", "пока", "до свидания"]):
            break

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Daryna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Daryna\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


JARVIS: Система активирована. Чем могу помочь?

Слушаю...
Вы: Луну
JARVIS: Я пока не знаю ответа на этот вопрос, но вы можете попросить меня найти его на Википедии!

Слушаю...

Слушаю...
Вы: Найди Википедии телефон
JARVIS: Ищу на Википедии: телефон

Слушаю...
Вы: Найди в Википедии звезда
JARVIS: Ищу на Википедии: звезда

Слушаю...
Вы: Найди Википедии Ёлка
JARVIS: Ищу на Википедии: ёлка

Слушаю...
Вы: Найди в Википедии кот
JARVIS: Ищу на Википедии: кот

Слушаю...
Вы: Найди в Википедии
JARVIS: Открываю главную страницу Википедии.

Слушаю...

Слушаю...
Вы: до свидания
JARVIS: Пока, прощай мой прекрасный светлый господин!
